In [1]:
import matplotlib
from numpy.distutils.system_info import lapack_opt_info
from scipy.fft import fft, ifft, fftfreq
import cv2
import numpy as np
import os
matplotlib.use('TkAgg')
from matplotlib import pyplot as plt
from scipy.signal import find_peaks
from scipy.signal import hilbert

C:\Users\mojtaba\AppData\Local\Temp\ipykernel_5360\3033587183.py:2: DeprecationWarning: 

  `numpy.distutils` is deprecated since NumPy 1.23.0, as a result
  of the deprecation of `distutils` itself. It will be removed for
  Python >= 3.12. For older Python versions it will remain present.
  It is recommended to use `setuptools < 60.0` for those Python versions.
  For more details, see:
    https://numpy.org/devdocs/reference/distutils_status_migration.html 


  from numpy.distutils.system_info import lapack_opt_info


In [2]:
# Load the signal
signal = np.load("all_intensity_value.npy").flatten()

In [3]:
N = len(signal)
T = 1.0  # Sampling interval (assume 1 unit per frame)

# FFT
yf = fft(signal)
xf = fftfreq(N, T)

# Create a mask for the frequency range 1.45 to 1.55
start_freq=0.01
end_freq=0.05
mask = (np.abs(xf) >= start_freq) & (np.abs(xf) <= end_freq)
filtered_yf = np.zeros_like(yf)
filtered_yf[mask] = yf[mask]

# Inverse FFT to get the reconstructed signal
reconstructed_signal = ifft(filtered_yf).real

# Normalize both signals
normalized_original = (signal - np.min(signal)) / (np.max(signal) - np.min(signal))
normalized_reconstructed = (reconstructed_signal - np.min(reconstructed_signal)) / (np.max(reconstructed_signal) - np.min(reconstructed_signal))

# Plot both signals
plt.figure(figsize=(12, 6))
plt.plot(normalized_original, label="Original Signal (Normalized)")
plt.plot(normalized_reconstructed, label=f"Reconstructed Signal ({start_freq}–{end_freq}  Hz)", linestyle='--')
plt.legend()
plt.title(f"Comparison of Original and Reconstructed Signal ({start_freq}–{end_freq} Hz Component)")
plt.xlabel("Frame Number")
plt.ylabel("Normalized Intensity")
plt.grid(True)
plt.tight_layout()
plt.show()

In [4]:
# Define the frequency range for low-pass filtering
start_freq = 0.01
end_freq = 0.03

# FFT of the signal
yf = fft(signal)
xf = fftfreq(N, T)

# Apply the mask to keep only the low frequencies (0 to 0.03 Hz)
mask = (np.abs(xf) >= start_freq) & (np.abs(xf) <= end_freq)
filtered_yf = np.zeros_like(yf)
filtered_yf[mask] = yf[mask]

# Inverse FFT to get the low-pass filtered signal (reconstructed trend)
low_pass_signal = ifft(filtered_yf).real


# Normalize both signals
signal= (signal - np.min(signal)) / (np.max(signal) - np.min(signal))
low_pass_signal1=low_pass_signal.copy()
low_pass_signal = (low_pass_signal - np.min(low_pass_signal)) / (np.max(low_pass_signal) - np.min(low_pass_signal))



# Difference between original and low-pass signal
diff = signal - low_pass_signal

# Define threshold (3 standard deviations as heuristic)
threshold = 1.5* np.std(diff)

# Identify outlier indices
outlier_indices = np.where(np.abs(diff) > threshold)[0]

# Plotting
plt.figure(figsize=(14, 6))

# Plot original signal
plt.plot(signal, label="Original Signal", alpha=0.6)

# Plot low-pass filtered signal
plt.plot(low_pass_signal, label="Low-Pass Reconstructed Signal", linewidth=2)

# Plot outliers as red dots on original signal
plt.scatter(outlier_indices, signal[outlier_indices], color='red', label="Outliers (due to interference)", zorder=5)

plt.title("Original vs Reconstructed Signal with Highlighted Interference Points")
plt.xlabel("Frame Number")
plt.ylabel("Intensity")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [5]:
# Create list of (frame_number, intensity) for all frames
all_data = np.column_stack((np.arange(N), signal))

# Remove the outlier frames
cleaned_data = np.delete(all_data, outlier_indices, axis=0)

# Plot original signal and cleaned signal
plt.figure(figsize=(14, 6))

# Original signal
plt.plot(all_data[:, 0], all_data[:, 1],'b.', label="Original Signal", alpha=0.4)

# Cleaned signal
plt.plot(cleaned_data[:, 0], cleaned_data[:, 1],'ro', label="Cleaned Signal (Outliers Removed)", linewidth=2)

plt.title("Original vs Cleaned Signal After Removing Interference Frames")
plt.xlabel("Frame Number")
plt.ylabel("Intensity")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Return the list of removed frame numbers
outlier_indices_list = outlier_indices.tolist()
outlier_indices_list[:10]  # Show first 10 as a preview


[162, 226, 342, 405, 543, 553, 648, 666, 671, 673]

In [6]:
np.save('cleaned_data.npy', cleaned_data)
np.save('low_pass_signal1.npy', low_pass_signal1)
np.save('low_pass_signal.npy', low_pass_signal)

In [7]:
plt.plot(low_pass_signal1)
plt.show()

In [8]:
bandpassed_signal = low_pass_signal1
# اعمال تبدیل هیلبرت روی سیگنال بازسازی‌شده باندی
analytic_signal = hilbert(bandpassed_signal)

instantaneous_phase = np.angle(analytic_signal)  # فاز لحظه‌ای در بازه [-π, π]

In [12]:
# توابع کمکی برای یافتن نزدیک‌ترین فریم به هر سیکل از یک فاز خاص
def extract_phase_points(phase_array, signal, target_phase, label, atol=0.05, min_distance=10):
    """
    از بین نقاطی که فاز لحظه‌ای نزدیک به مقدار خاصی هستند،
    فقط یکی از هر گروه متوالی را نگه می‌دارد.
    """
    indices = np.where(np.isclose(phase_array, target_phase, atol=atol))[0]
    if label in ['π', '-π']:
        indices = np.where((phase_array > 3.) | (phase_array < -3.))[0]

    # حذف نقاط خیلی نزدیک به هم
    selected = []
    prev = -min_distance
    for idx in indices:
        if idx - prev >= min_distance:
            selected.append(idx)
            prev = idx
    return np.array(selected)

# تعریف فازها و ویژگی‌هایشان
phase_targets = {'0': 0, 'π/2': np.pi/2, 'π': np.pi, '3π/2': -np.pi/2}
phase_colors = {'0': 'k', 'π/2': 'r', 'π': 'b', '3π/2': 'g'}
phase_markers = {'0': 'x', 'π/2': '^', 'π': 'o', '3π/2': 's'}

# استخراج نقاط نماینده هر فاز
phase_points = {}
for label, value in phase_targets.items():
    points = extract_phase_points(instantaneous_phase, bandpassed_signal, value, label)
    phase_points[label] = points

# رسم نتایج روی نمودار
plt.figure(figsize=(15, 5))
plt.plot(bandpassed_signal, label='Bandpassed Signal', linewidth=1.5)

for label in phase_targets:
    if label == 'π' :
        valleys, _ = find_peaks(-bandpassed_signal, distance=10)

        # انتخاب دره‌هایی که در محدوده‌ی فاز π (یا -π) طبق هیلبرت هستند
        pi_region = (instantaneous_phase > 2.8) | (instantaneous_phase < -2.8)
        pts = [v for v in valleys if pi_region[v]]
    else:

        pts = phase_points[label]

    plt.plot(pts, bandpassed_signal[pts], linestyle='None',
             marker=phase_markers[label], color=phase_colors[label],
             label=f'Phase ≈ {label}')

plt.title("Bandpassed Signal with Representative Phase Points")
plt.xlabel("Frame Index")
plt.ylabel("Intensity")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [10]:
# Normalize cleaned_data intensity values and low_pass_signal for comparability
cleaned_frames = cleaned_data[:, 0].astype(int)
cleaned_intensity = cleaned_data[:, 1]
norm_cleaned_intensity = (cleaned_intensity - np.min(cleaned_intensity)) / (np.max(cleaned_intensity) - np.min(cleaned_intensity))
norm_low_pass_signal = (low_pass_signal - np.min(low_pass_signal)) / (np.max(low_pass_signal) - np.min(low_pass_signal))

# Define threshold for similarity
similarity_threshold = 0.05  # Can be tuned

# Dictionary to store matched phase points in cleaned signal
matched_phases_cleaned = {label: [] for label in phase_points}

# Try to match each phase point from low-pass signal to closest frame in cleaned_data
shifts=[0,-1,1,-2,2,-3,3,-4,4,-5,5,-6,6]
for label, indices in phase_points.items():
    for idx in indices:
        if idx >= len(norm_low_pass_signal):
            continue
        target_intensity = norm_low_pass_signal[idx]

        # Find cleaned frame closest in index
        if idx in cleaned_frames:
            i = np.where(cleaned_frames == idx)[0][0]
            candidate = norm_cleaned_intensity[i]
            if abs(candidate - target_intensity) <= similarity_threshold:
                matched_phases_cleaned[label].append(cleaned_frames[i])
                continue

        # Try previous and next neighbors if available


        for shift in shifts:

            neighbor_idx = idx + shift
            print(neighbor_idx )
            if neighbor_idx in cleaned_frames:
                i = np.where(cleaned_frames == neighbor_idx)[0][0]
                candidate = norm_cleaned_intensity[i]
                if abs(candidate - target_intensity) <= similarity_threshold:
                    matched_phases_cleaned[label].append(cleaned_frames[i])
                    break



# Plotting original low-pass signal and cleaned signal with phases
plt.figure(figsize=(15, 6))
plt.plot(norm_low_pass_signal, label='Low-pass Signal (Normalized)', linewidth=1.2, alpha=0.7)
plt.plot(cleaned_frames, norm_cleaned_intensity, label='Cleaned Signal (Normalized)', linewidth=1.5)

# Plot phases for low-pass signal
for label in phase_points:
    if label == 'π':
        valleys, _ = find_peaks(-norm_low_pass_signal, distance=10)
        pi_region = (instantaneous_phase > 2.8) | (instantaneous_phase < -2.8)
        pts = [v for v in valleys if pi_region[v]]
    else:
        pts = phase_points[label]
    plt.plot(pts, norm_low_pass_signal[pts], linestyle='None',
             marker=phase_markers[label], color=phase_colors[label], label=f'{label} (Low-pass)')

# Plot matched phases in cleaned signal
for label, frames in matched_phases_cleaned.items():
    intensities = [norm_cleaned_intensity[np.where(cleaned_frames == f)[0][0]] for f in frames]
    plt.plot(frames, intensities, linestyle='None', marker=phase_markers[label],
             color=phase_colors[label], label=f'{label} (Cleaned)', markerfacecolor='none')

plt.title("Comparison of Phase Points in Low-pass and Cleaned Signals")
plt.xlabel("Frame Index")
plt.ylabel("Normalized Intensity")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


9
8
58
57
59
161
160
162
159
163
337
336
403
402
470
469
545
544
546
686
685
687
684
736
735
880
879
881
981
980
1033
1032
1034
1031
1035
1030
1036
1102
1101
1103
1100
1104
1099
1105
1098
1173
1172
1174
1171
1243
1242
1244
1241
1245
1240
1246
1239
1247
1295
1294
1371
1370
1433
1432
1434
1431
1435
1430
1436
1909
1908
1910
2120
2119
2255
2254
2256
2253
2257
2252
2323
2322
2324
2321
2389
2388
2463
2462
2464
2461
2465
2460
2466
2459
2467
2458
2468
2457
2469
2534
2533
2535
2532
2536
2531
2537
2530
2538
2529
2539
2613
2612
2614
2611
2615
2610
2616
2609
2617
2608
2618
2748
2747
2849
2848
2850
2847
2851
2846
2852
2845
2853
2929
2928
3029
3028
3030
3027
3031
3026
3235
3234
3236
96
95
175
174
297
296
747
746
893
892
918
917
919
916
920
915
921
1047
1046
1048
1117
1116
1188
1187
1189
1318
1317
1513
1512
1654
1653
1655
1652
1656
1719
1718
1720
1717
1856
1855
1857
1854
1858
1853
1859
1852
1860
1851
1861
1850
1862
1925
1924
1993
1992
1994
1991
2067
2066
2136
2135
2204
2203
2205
2202
2206
2340
2339
2

In [11]:
print(len(matched_phases_cleaned['0']),
        len(matched_phases_cleaned['π/2']),

            len(matched_phases_cleaned['π']),

            len(matched_phases_cleaned['3π/2']))



55 53 60 49


In [14]:
print((matched_phases_cleaned['0'][0]),
        (matched_phases_cleaned['π/2'][0]),

            (matched_phases_cleaned['π'][0]),

            (matched_phases_cleaned['3π/2'][0]))


8 20 30 44


In [15]:
# Step 1: Combine all matched phase points into a unified list with labels
combined_phases = []
for label, frames in matched_phases_cleaned.items():
    for frame in frames:
        combined_phases.append((frame, label))

# Step 2: Sort combined list by frame index
combined_phases.sort()

# Define expected sequence of phase labels
expected_sequence = ['0', 'π/2', 'π', '3π/2']
valid_cycles = []

# Step 3: Iterate through sorted phases to find valid consecutive phase groups
i = 0
while i <= len(combined_phases) - 4:
    labels_window = [combined_phases[i + j][1] for j in range(4)]
    frames_window = [combined_phases[i + j][0] for j in range(4)]

    # Check if the labels match the expected order
    if labels_window == expected_sequence:
        # Get normalized intensities for frame 0 and π
        f0, fpi = frames_window[0], frames_window[2]
        try:
            i0 = np.where(cleaned_frames == f0)[0][0]
            ipi = np.where(cleaned_frames == fpi)[0][0]
            intensity_0 = norm_cleaned_intensity[i0]
            intensity_pi = norm_cleaned_intensity[ipi]
        except IndexError:
            i += 1
            continue

        # Check if the intensity difference is large enough
        if intensity_0 - intensity_pi > 0.05:
            valid_cycles.append(frames_window)
            i += 4  # Move to next possible cycle after this one
        else:
            i += 1  # Not valid, slide by 1
    else:
        # If sequence is broken or has duplicates, move ahead to the next starting phase
        i += 1

valid_cycles[:5]  # Preview the first 5 valid phase cycles


[[8, 20, 30, 44],
 [336, 350, 363, 383],
 [469, 491, 511, 530],
 [903, 921, 929, 943],
 [980, 994, 1007, 1020]]

In [16]:
# Plot the cleaned signal
plt.figure(figsize=(15, 6))
plt.plot(cleaned_frames, norm_cleaned_intensity, label='Cleaned Signal (Normalized)', color='orange', linewidth=1.5)

# Overlay valid phase cycles on the plot
for cycle in valid_cycles:
    cycle_intensities = [norm_cleaned_intensity[np.where(cleaned_frames == f)[0][0]] for f in cycle]
    labels = ['0', 'π/2', 'π', '3π/2']
    for f, inten, lbl in zip(cycle, cycle_intensities, labels):
        plt.plot(f, inten, marker=phase_markers[lbl], color=phase_colors[lbl], markersize=8,
                 label=f'{lbl} phase' if cycle == valid_cycles[0] else "", linestyle='None')

plt.title("Valid Phase Cycles on Cleaned Signal")
plt.xlabel("Frame Index")
plt.ylabel("Normalized Intensity")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()
